# Chapter 4: Data Cleaning & Transformation

Real-world data is messy. This chapter covers:

1. Handling missing data
2. Duplicates
3. Data type conversions
4. String methods (`.str` accessor)
5. `apply`, `map`, `applymap`/`map`
6. `rename`, `replace`
7. Binning: `pd.cut`, `pd.qcut`

In [ ]:
import pandas as pd
import numpy as np

## 4.1 Handling Missing Data

In [ ]:
df = pd.DataFrame({
    'name': ['Alice', 'Bob', None, 'Diana', 'Eve'],
    'age': [25, np.nan, 35, 28, np.nan],
    'salary': [70000, 80000, np.nan, 75000, 85000],
    'city': ['NYC', 'London', 'Paris', None, 'Sydney'],
})
df

In [ ]:
# Detect missing values
print('--- isna() ---')
print(df.isna())
print()
print('Missing count per column:')
print(df.isna().sum())
print()
print('Missing percentage:')
print((df.isna().mean() * 100).round(1))

In [ ]:
# notna() — opposite of isna()
df[df['age'].notna()]

In [ ]:
# dropna — remove rows with any NaN
print('dropna():\n', df.dropna())
print()

# Drop rows where specific columns are NaN
print('dropna(subset=["age"]):\n', df.dropna(subset=['age']))
print()

# Drop only if ALL values are NaN
print('dropna(how="all"):\n', df.dropna(how='all'))

In [ ]:
# fillna — fill missing values
print('Fill with 0:\n', df['age'].fillna(0))
print()

# Forward fill
print('Forward fill:\n', df['age'].ffill())
print()

# Fill with mean
print('Fill with mean:\n', df['age'].fillna(df['age'].mean()))

In [ ]:
# interpolate — linear interpolation
s = pd.Series([1, np.nan, np.nan, 4, 5, np.nan, 7])
print('Original: ', s.values)
print('Interpolated:', s.interpolate().values)

## 4.2 Handling Duplicates

In [ ]:
df_dup = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob'],
    'score': [90, 85, 90, 78, 85],
})

print('Duplicated rows:')
print(df_dup.duplicated())
print()
print(f'Number of duplicates: {df_dup.duplicated().sum()}')

In [ ]:
# Drop duplicates
df_dup.drop_duplicates()

In [ ]:
# Keep last occurrence
df_dup.drop_duplicates(keep='last')

In [ ]:
# Check duplicates on specific columns only
df_dup.drop_duplicates(subset=['name'], keep='first')

## 4.3 Data Type Conversions

In [ ]:
df_types = pd.DataFrame({
    'id': ['1', '2', '3', '4'],
    'price': ['10.5', '20.3', 'N/A', '15.0'],
    'date_str': ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05'],
})
print(df_types.dtypes)

In [ ]:
# astype — explicit conversion
df_types['id'] = df_types['id'].astype(int)
print(df_types.dtypes)

In [ ]:
# pd.to_numeric — handles errors gracefully
df_types['price'] = pd.to_numeric(df_types['price'], errors='coerce')
df_types

In [ ]:
# pd.to_datetime
df_types['date'] = pd.to_datetime(df_types['date_str'])
df_types.dtypes

## 4.4 String Methods — The `.str` Accessor

Any Series of dtype `object` (strings) exposes `.str` for vectorized string operations.

In [ ]:
names = pd.Series(['  Alice Smith  ', 'BOB jones', 'charlie BROWN', 'diana Prince'])

print('lower:\n', names.str.lower().values)
print('upper:\n', names.str.upper().values)
print('title:\n', names.str.title().values)
print('strip:\n', names.str.strip().values)
print('len:  \n', names.str.len().values)

In [ ]:
# Split
print('split:')
print(names.str.strip().str.split(' '))

# Get first name
print('\nFirst names:', names.str.strip().str.split(' ').str[0].values)

In [ ]:
# contains (regex by default)
print('Contains "o":', names.str.contains('o', case=False).values)

# replace
print('Replace:', names.str.strip().str.replace('Smith', 'Johnson', regex=False).values)

In [ ]:
# extract — extract regex groups
emails = pd.Series(['alice@gmail.com', 'bob@yahoo.com', 'charlie@outlook.com'])
emails.str.extract(r'(\w+)@(\w+)\.com')

## 4.5 `apply`, `map`, and `applymap`/`map`

In [ ]:
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'score_a': [85, 90, 78],
    'score_b': [92, 88, 95],
})

In [ ]:
# Series.apply — apply a function element-wise to a Series
def grade(score):
    if score >= 90: return 'A'
    if score >= 80: return 'B'
    if score >= 70: return 'C'
    return 'D'

df['grade_a'] = df['score_a'].apply(grade)
df

In [ ]:
# Series.map — map values using a dict or function
grade_map = {'A': 'Excellent', 'B': 'Good', 'C': 'Average', 'D': 'Poor'}
df['grade_label'] = df['grade_a'].map(grade_map)
df

In [ ]:
# DataFrame.apply — apply along an axis
# axis=1 means apply function to each ROW
df['avg_score'] = df[['score_a', 'score_b']].apply(np.mean, axis=1)
df

In [ ]:
# DataFrame.map (Pandas 2.1+) or applymap (older)
# Element-wise operation on entire DataFrame
numeric_df = df[['score_a', 'score_b']]

# Works in Pandas >= 2.1
try:
    result = numeric_df.map(lambda x: f'{x}%')
except AttributeError:
    result = numeric_df.applymap(lambda x: f'{x}%')
result

## 4.6 `rename` & `replace`

In [ ]:
# Rename columns
df_renamed = df.rename(columns={
    'score_a': 'math_score',
    'score_b': 'science_score'
})
df_renamed.columns.tolist()

In [ ]:
# Replace values in a column
df_copy = df.copy()
df_copy['grade_a'] = df_copy['grade_a'].replace({'A': 'A+', 'B': 'B+'})
df_copy[['name', 'grade_a']]

## 4.7 Binning: `pd.cut` & `pd.qcut`

In [ ]:
ages = pd.Series([15, 22, 35, 45, 55, 67, 78, 25, 33, 50])

# pd.cut — equal-width bins
bins = [0, 18, 35, 55, 100]
labels = ['Minor', 'Young Adult', 'Middle Age', 'Senior']

age_groups = pd.cut(ages, bins=bins, labels=labels)
print('pd.cut (equal width):')
print(pd.DataFrame({'age': ages, 'group': age_groups}))

In [ ]:
# pd.qcut — equal-frequency bins (quantile-based)
quartiles = pd.qcut(ages, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print('pd.qcut (equal frequency):')
print(pd.DataFrame({'age': ages, 'quartile': quartiles}))
print()
print('Value counts:')
print(quartiles.value_counts().sort_index())

## 📝 Key Takeaways

| Topic | Key Point |
|-------|----------|
| **Missing data** | `isna()`, `fillna()`, `dropna()`, `interpolate()` |
| **Duplicates** | `duplicated()`, `drop_duplicates()` |
| **Type conversion** | `astype()`, `pd.to_numeric(errors='coerce')` |
| **Strings** | `.str` accessor for vectorized string ops |
| **apply/map** | `apply` for custom functions; `map` for dict lookups |
| **Binning** | `pd.cut` (equal width) vs `pd.qcut` (equal frequency) |